# ノートブック1：動画からロボットのお手本へ（ランタイム：T4 GPU）【改訂版】

「手順書_案A_ヒューマノイドにダンスを真似させる.md」の手順2をまとめたノートブックです。上から順に実行してください。

**旧版からの主な変更**
- 元動画を **30fps に変換**してから推定（29.97fps のままだと 2 分で約 0.14 秒、音とずれる）
- 姿勢推定は **2分19秒の全編に 1 回だけ**。20 秒版は全編の結果から切り出す（20 秒版と 2 分版のお手本が一致する）
- 推定ミスの除去 → ゼロ位相フィルタ → 接地区間ごとの高さ補正、に変更
- **物理チェック（2-7）** を追加：学習の前に「物理的に無理な秒」を洗い出す
- 各リポジトリを**コミット固定**で取得

## 2-1. ドライブの接続と共通設定

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
P = '/content/drive/MyDrive/pai_final'
os.environ['P'] = P

# 使うリポジトリのコミット（この版で動作を確認したソースに固定。切断後に入れ直しても同じ版になる）
PINS = {
    'GVHMR':            ('https://github.com/zju3dv/GVHMR',                  'ee960bb6e2ea2d381aa97f08e9b71ef320b624b1'),
    'GMR':              ('https://github.com/YanjieZe/GMR',                  'bb1bbe40774794fceb2a7c579a3464a28e68c844'),
    'unitree_rl_mjlab': ('https://github.com/unitreerobotics/unitree_rl_mjlab', '1425b15f73bd4095f0df53709d7c389c3eb9e790'),
    'mjlab':            ('https://github.com/mujocolab/mjlab',               '5af32e378dcb93c9e881ace83cc5a3f5d373fe60'),  # v1.2.0
}
for k, (url, rev) in PINS.items():
    os.environ[f'URL_{k}'], os.environ[f'REV_{k}'] = url, rev

# サビ後半 20 秒が、全編（伊原六花さんの動画 2分19秒）の何秒目か（音声の照合で特定済み）
SEG_START, SEG_END = 59.815, 80.78
os.environ['SEG_START'], os.environ['SEG_END'] = str(SEG_START), str(SEG_END)

# MuJoCo を画面なし（EGL）で描画するための設定
NVIDIA_ICD = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD):
    os.makedirs(os.path.dirname(NVIDIA_ICD), exist_ok=True)
    with open(NVIDIA_ICD, 'w') as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTHONUNBUFFERED'] = '1'   # ログをすぐファイルに書き出す

# 動画・画像をノートブック上で表示する関数
from base64 import b64encode
from IPython.display import HTML, Image, display
def show(path, width=720):
    data = b64encode(open(path, 'rb').read()).decode()
    return HTML(f'<video width={width} controls loop><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

os.environ['FULL'] = 'the_feels_full'   # 全編（2分19秒）
os.environ['SEG'] = 'the_feels_20s'     # サビ後半 20 秒

!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls -la $P/input $P/scripts
!ls -R $P/models | head -20

## 2-2. GVHMR のインストール（15〜25分）

In [ ]:
%%bash
set -e
pip -q install uv
cd /content
bash $P/scripts/pin_clone.sh $URL_GVHMR /content/GVHMR $REV_GVHMR
uv venv /content/venv_gvhmr --python 3.10 --seed -q
PY=/content/venv_gvhmr/bin/python
# ビルド時に numpy / pip を必要とするパッケージを先に入れる
$PY -m pip -q install "numpy==1.23.5" cython "setuptools>=68" wheel
$PY -m pip -q install --no-build-isolation chumpy cython_bbox
cd GVHMR
$PY -m pip -q install -r requirements.txt
$PY -m pip -q install -e .
$PY -c "import torch, pytorch3d; print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())"

## 2-3. 学習済みモデルと人体モデルを配置

In [ ]:
%%bash
set -e
cd /content/GVHMR
mkdir -p inputs/checkpoints/body_models outputs
cp -r $P/models/smpl $P/models/smplx inputs/checkpoints/body_models/
cp -r $P/gvhmr_ckpt/* inputs/checkpoints/
find inputs/checkpoints -type f | sort

## 2-4. 元動画を 30fps に変換（1分）

元動画は 29.97fps です。GMR は GVHMR の結果を **30fps と決め打ち**で読むため、そのままだと 2 分で約 0.14 秒、ダンスが音より先に進みます。
ここで 30fps ちょうどに変換しておけば、以降の時刻（秒）はすべて元動画と一致します。

In [ ]:
%%bash
set -e
cd $P/input
ffmpeg -nostdin -y -v error -i the_feels_full_src.mp4 -vf fps=30 -c:v libx264 -crf 18 -preset fast -c:a copy $FULL.mp4
ffprobe -v error -select_streams v:0 -show_entries stream=r_frame_rate,nb_frames:format=duration -of compact $FULL.mp4 < /dev/null

## 2-5. 全編から人の3D姿勢を推定（10〜20分）

固定カメラの動画なので `-s`（カメラ静止）を付けます。

GVHMR は最後に「元動画＋推定結果」を横に並べた確認用動画を作ります（`[Merge Videos]`）。2分19秒あると進捗表示なしで長時間止まって見えるうえ、以降の処理には不要なので、先に空ファイルを置いて省略しています。
済んだ段階はスキップされるので、途中で止まった場合もこのセルをもう一度実行すれば続きから進みます。

In [ ]:
%%bash
set -e
cd /content/GVHMR
OUT=outputs/demo/$FULL
mkdir -p $OUT
touch $OUT/${FULL}_3_incam_global_horiz.mp4   # 横並びの確認用動画（不要・時間がかかる）を省略
/content/venv_gvhmr/bin/python tools/demo/demo.py --video=$P/input/$FULL.mp4 -s < /dev/null
# 後の手順で使うものだけドライブへコピー
mkdir -p $P/outputs/gvhmr/$FULL
cp $OUT/hmr4d_results.pt $OUT/1_incam.mp4 $OUT/2_global.mp4 $P/outputs/gvhmr/$FULL/
ls -la $P/outputs/gvhmr/$FULL


In [ ]:
# 推定結果を確認（人型メッシュだけの動画。元動画の人物は映らない）。速いターンで前後が反転していないか見る
show(f"{P}/outputs/gvhmr/{os.environ['FULL']}/2_global.mp4")

## 2-6. GMR と、物理チェック用の G1 モデル（mjlab）を用意

In [ ]:
%%bash
set -e
cd /content
bash $P/scripts/pin_clone.sh $URL_GMR /content/GMR $REV_GMR
bash $P/scripts/pin_clone.sh $URL_mjlab /content/mjlab $REV_mjlab   # 学習で使う G1 のモデル (xml) だけ使う
uv venv /content/venv_gmr --python 3.10 --seed -q
PY=/content/venv_gmr/bin/python
$PY -m pip -q install torch --index-url https://download.pytorch.org/whl/cpu   # GMR は CPU で十分
cd GMR
$PY -m pip -q install -e .
$PY -m pip -q install scipy matplotlib "imageio[ffmpeg]"
mkdir -p assets/body_models/smplx
cp $P/models/smplx/SMPLX_*.npz assets/body_models/smplx/
$PY -c "import mujoco; print('mujoco', mujoco.__version__)"

## 2-7. お手本（CSV）を作る

`gvhmr_to_g1_csv.py` が行うこと：
1. GMR で G1 の関節角に変換（全編。加工前の結果を `the_feels_full_raw.csv` に保存）
2. 推定ミス（1〜数フレームの姿勢の飛び）を検出して補間
3. ゼロ位相 Butterworth フィルタで平滑化（関節 7Hz・根 4Hz。位相が遅れないので音ハメが崩れない）
4. 接地区間ごとに根の高さを補正し、足滑りを測る
5. CSV・プレビュー動画・診断レポート（`*_prep_report.json`）を保存

20 秒版は、全編の `_raw.csv` から切り出して作ります（IK を解き直さないので速い）。

In [ ]:
%%bash
set -e
cd /content/GMR
PY=/content/venv_gmr/bin/python
# 全編（2分19秒）
$PY $P/scripts/gvhmr_to_g1_csv.py --gvhmr_pred_file $P/outputs/gvhmr/$FULL/hmr4d_results.pt \
  --out_dir $P/outputs/motions --name $FULL
# サビ後半 20 秒（全編の raw から切り出し）
$PY $P/scripts/gvhmr_to_g1_csv.py --from_csv $P/outputs/motions/${FULL}_raw.csv \
  --out_dir $P/outputs/motions --name $SEG --start $SEG_START --end $SEG_END

In [ ]:
# お手本（物理なし）のプレビュー（20 秒版）
show(f"{P}/outputs/motions/{os.environ['SEG']}_retarget_preview.mp4")

## 2-8. 物理チェック（学習の前に必ず実行）

学習に使うのと同じ G1 のモデルで、お手本の各フレームを調べます。

| 項目 | 意味 | 多いときの対処 |
|---|---|---|
| `hover` | 両足が浮いているのに上向きの力が必要（空中で止まる） | 根の高さの推定ミス。`--cut_root_z 3` に下げる、`--contact_h` を上げる |
| `pull` | 下向きの力が必要（床は引っぱれない） | 同上。急な沈み込みの推定ミス |
| `slip` | 必要な水平力が摩擦の限界を超える | 足滑り・急な重心移動。その区間の動きを小さくする |
| `self_collision` | 腕と胴体などが 5mm 以上めり込む | 振付で手が体に触れる区間。学習では大きな罰になる |
| `joint_limit` / `joint_vel` | 可動域外・速度上限超え | 推定ミスか、G1 では無理な動き |

結果の「難所（問題が 3 フレーム以上ある秒）」は、学習で転びやすい場所の予想です。学習後の評価と照らし合わせると、レポートの考察になります。

In [ ]:
%%bash
set -e
XML=/content/mjlab/src/mjlab/asset_zoo/robots/unitree_g1/xmls/g1.xml
PY=/content/venv_gmr/bin/python
if [ ! -x $PY ] || [ ! -f $XML ]; then
  echo '[stop] ランタイムが切断されて GMR / mjlab が消えています。先に 2-1 と 2-6 を実行してください。'
  echo '       （チェック結果がドライブに保存済みなら、このセルは飛ばして次のグラフのセルだけ実行すれば OK）'
  ls $P/outputs/check/*_check_frames.csv 2>/dev/null
  exit 1
fi
mkdir -p $P/outputs/check
$PY $P/scripts/check_reference.py --csv $P/outputs/motions/$SEG.csv --robot_xml $XML \
  --t0 $SEG_START --out_prefix $P/outputs/check/$SEG
$PY $P/scripts/check_reference.py --csv $P/outputs/motions/$FULL.csv --robot_xml $XML \
  --out_prefix $P/outputs/check/$FULL > $P/outputs/check/${FULL}_log.txt
head -40 $P/outputs/check/${FULL}_log.txt   # 全編は長いので先頭だけ表示（全文は _log.txt）

In [ ]:
# 物理チェックのグラフ（保存済みの _check_frames.csv から、このノートブック上で描く）
import numpy as np, matplotlib.pyplot as plt
def plot_check(name, contact_h=0.02):
    path = f"{P}/outputs/check/{name}_check_frames.csv"
    D = np.loadtxt(path, delimiter=",", skiprows=1)
    cols = open(path).readline().strip().split(",")
    t, fz, fxy = D[:, 0], D[:, 1], D[:, 2]
    air = D[:, 3:5].min(1) >= contact_h
    flags = {n: D[:, i].astype(bool) for i, n in enumerate(cols) if i >= 6}
    fig, ax = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1.4]})
    ax[0].plot(t, fz, lw=1, label="required vertical force / mg")
    ax[0].plot(t, fxy, lw=1, label="required horizontal force / mg")
    ax[0].fill_between(t, -1, 4, where=air, color="0.85", label="both feet airborne")
    ax[0].axhline(0, color="k", lw=0.5); ax[0].set_ylim(-1, 4); ax[0].legend(loc="upper right", fontsize=8)
    for k, (n, v) in enumerate(flags.items()):
        ax[1].scatter(t[v], np.full(v.sum(), k), s=4)
    ax[1].set_yticks(range(len(flags))); ax[1].set_yticklabels(list(flags)); ax[1].set_xlabel("time [s]")
    ax[0].set_title(name); fig.tight_layout()
    fig.savefig(f"{P}/outputs/check/{name}_check.png", dpi=110); plt.show()

plot_check(os.environ['SEG'])
plot_check(os.environ['FULL'])


### （任意）平滑化の比較実験 — レポートの「工夫点」用

旧版の移動平均（窓 5）と今回のフィルタで、物理チェックの結果を比べます。

In [ ]:
%%bash
set -e
cd /content/GMR
PY=/content/venv_gmr/bin/python
XML=/content/mjlab/src/mjlab/asset_zoo/robots/unitree_g1/xmls/g1.xml
$PY $P/scripts/gvhmr_to_g1_csv.py --from_csv $P/outputs/motions/${FULL}_raw.csv --out_dir $P/outputs/motions \
  --name ${SEG}_ma5 --start $SEG_START --end $SEG_END --no_outlier --filter ma --smooth 5 --ground constant --no_video
$PY $P/scripts/check_reference.py --csv $P/outputs/motions/${SEG}_ma5.csv --robot_xml $XML \
  --t0 $SEG_START --out_prefix $P/outputs/check/${SEG}_ma5 > /dev/null
python3 - <<EOF
import json
for n in ["${SEG}_ma5", "${SEG}"]:
    j = json.load(open(f"$P/outputs/check/{n}_check.json"))
    print(f"{n:22s}", j["frames_flagged"])
EOF

ここまで終わったら、ランタイムを切断します（「ランタイム」→「ランタイムを接続解除して削除」）。成果物はすべてドライブに保存されています。